In [1]:
!pip install pyspark -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ETL_Transform") \
    .getOrCreate()

print("✅ Spark démarré :", spark.version)

✅ Spark démarré : 4.0.2


In [5]:
df = spark.read.parquet('/content/drive/MyDrive/ETL_projet/bronze/')

print("✅ Bronze chargé :", df.count(), "lignes")
df.printSchema()

✅ Bronze chargé : 1067371 lignes
root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)



In [6]:
from pyspark.sql.functions import col

df = df.dropna(subset=['Customer ID'])
df = df.dropDuplicates()
df = df.filter(~col('Invoice').startswith('C'))
df = df.filter((col('Price') > 0) & (col('Quantity') > 0))

print("✅ Après nettoyage :", df.count(), "lignes")

✅ Après nettoyage : 779425 lignes


In [7]:
df = df.withColumn('TotalPrice', col('Quantity') * col('Price'))

df.write.mode("overwrite").parquet(
    '/content/drive/MyDrive/ETL_projet/silver/'
)

print("✅ Silver sauvegardé !")
df.show(3)

✅ Silver sauvegardé !
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|        TotalPrice|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------------+
| 489439|    22065|CHRISTMAS PUDDING...|      12|2009-12-01 09:28:00| 1.45|    12682.0|        France|              17.4|
| 489450|   85206A|CREAM FELT EASTER...|       6|2009-12-01 10:36:00| 1.65|    16321.0|     Australia| 9.899999999999999|
| 489531|    21587|COSY HOUR GIANT T...|       6|2009-12-01 11:57:00| 2.55|    14871.0|United Kingdom|15.299999999999999|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------------+
only showing top 3 rows
